# Homework 2: Vector Search

https://github.com/DataTalksClub/llm-zoomcamp/blob/main/cohorts/2026/02-vector-search/homework.md

I’ll be applying what I’ve learned in Module 2 by building a search system that uses both Vector and Keyword matching. 
I’ll be working with the same 72 course lesson files as module 1, and since my results might differ slightly from the provided options, I’ll just aim for the closest match.

## Setup and Load Data
In this step, I will import the necessary libraries, load the course data from GitHub (via the `gitsource` library in commit `8c1834d`), and initialize the `Embedder`.

In [2]:
from gitsource import GithubRepositoryDataReader
from embedder import Embedder
import numpy as np
import minsearch
from gitsource import chunk_documents

# Load data
reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda x: "/lessons/" in x
)
raw_docs = reader.read()
documents = [doc.parse() for doc in raw_docs]

print(f"Download successful {len(documents)} documents!")

# Initialize Embedder
embed = Embedder()

Download successful 72 documents!


## Q1. Embedding a query
Embed the query: *"How does approximate nearest neighbor search work?"*.  
**What is the first value (`v[0]`)?**

In [3]:
query_q1 = "How does approximate nearest neighbor search work?"
v_q1 = embed.encode(query_q1)
print(f"Q1 Answer: {v_q1[0]:.2f}")

Q1 Answer: -0.02


## Q2. Cosine similarity
Take the content of the page `02-vector-search/lessons/07-sqlitesearch-vector.md`, embed its `content`, and  **calculate the cosine similarity with the query vector from Q1?**

In [4]:
target_filename = "02-vector-search/lessons/07-sqlitesearch-vector.md"
target_doc = next(doc for doc in documents if doc["filename"] == target_filename)

v_doc = embed.encode(target_doc["content"])
similarity = np.dot(v_q1, v_doc)
print(f"Q2 Answer: {similarity:.2f}")

Q2 Answer: 0.36


## Q3. Chunking and search by hand
Chunk pages, embed the content of each chunk into a matrix `X`, and calculate the score for query Q1 using all chunks.  
**Which file does the highest-scoring chunk belong to (its filename)?**

In [5]:
chunks = chunk_documents(documents, size=2000, step=1000)

chunk_contents = [chunk["content"] for chunk in chunks]
X = np.array(embed.encode_batch(chunk_contents))

scores = X.dot(v_q1)
best_chunk_idx = np.argmax(scores)

print(f"Q3 Answer: {chunks[best_chunk_idx]['filename']}")

Q3 Answer: 02-vector-search/lessons/07-sqlitesearch-vector.md


## Q4. Vector search with minsearch
Use `VectorSearch` from `minsearch` to run a search for the sentence: *"What metric do we use to evaluate a search engine?"*  
**What is the filename of the first result?**

In [6]:
vindex = minsearch.VectorSearch(keyword_fields=["filename"])
vindex.fit(X, chunks)

query_q4 = "What metric do we use to evaluate a search engine?"
v_q4 = embed.encode(query_q4)
v_results = vindex.search(v_q4, num_results=1)

print(f"Q4 Answer: {v_results[0]['filename']}")

Q4 Answer: 04-evaluation/lessons/05-search-metrics.md


## Q5. Text search vs vector search
Index similar chunks using `Index` from minsearch with `content` as the text field. Run both text and vector search for the sentence: *"How do I store vectors in PostgreSQL?"*.  
**Which file appears in the top 5 of the vector results but NOT in the text results?**

In [7]:
tindex = minsearch.Index(text_fields=["content"], keyword_fields=["filename"])
tindex.fit(chunks)

query_q5 = "How do I store vectors in PostgreSQL?"
v_q5 = embed.encode(query_q5)

vec_top5 = vindex.search(v_q5, num_results=5)
text_top5 = tindex.search(query_q5, num_results=5)

vec_filenames = [res["filename"] for res in vec_top5]
text_filenames = [res["filename"] for res in text_top5]

difference = set(vec_filenames) - set(text_filenames)
print(f"Q5 Answer: {list(difference)}")

Q5 Answer: ['02-vector-search/lessons/08-pgvector.md']


## Q6. Hybrid search
Run the query *"How do I give the model access to tools?"* with vector and text search, then combine the top results using Reciprocal Rank Fusion (RRF) with $k = 60$.  
**Which file is ranked first after RRF?**

In [8]:
def rrf(search_results, k=60, num_results=10):
    scores = {}
    doc_map = {}
    
    for results in search_results:
        for rank, doc in enumerate(results):
            key = f"{doc['filename']}_{doc['start']}" 
            if key not in scores:
                scores[key] = 0
                doc_map[key] = doc
            scores[key] += 1 / (k + rank + 1)
            
    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    return [doc_map[key] for key, _ in ranked[:num_results]]

query_q6 = "How do I give the model access to tools?"
v_q6 = embed.encode(query_q6)

vec_results = vindex.search(v_q6, num_results=10)
text_results = tindex.search(query_q6, num_results=10)

hybrid_results = rrf([vec_results, text_results], k=60, num_results=1)

print(f"Q6 Answer: {hybrid_results[0]['filename']}")

Q6 Answer: 01-agentic-rag/lessons/13-function-calling.md
